# 模型

语言学习模型（LLM）是功能强大的AI工具，能够像人类一样理解和生成文本。它们用途广泛，无需针对每项任务进行专门训练，即可撰写内容、翻译语言、撰写摘要和回答问题。

除了文本生成之外，许多模型还支持：
- 工具调用- 调用外部工具（如数据库查询或 API 调用）并在其响应中使用结果。
- 结构化输出——模型的响应被限制在定义的格式内。
- 多模态——处理并返回除文本以外的数据，例如图像、音频和视频。
- 推理——模型执行多步骤推理以得出结论。

模型是智能体的推理引擎。它们驱动智能体的决策过程，决定调用哪些工具、如何解释结果以及何时给出最终答案。

您选择的模型的质量和功能直接影响智能体的可靠性和性能。不同的模型擅长不同的任务——有些模型更擅长执行复杂的指令，有些模型更擅长结构化推理，还有一些模型支持更大的上下文窗口以处理更多信息。

LangChain 的标准模型接口可让您访问许多不同的提供商集成，从而可以轻松地尝试和切换模型，以找到最适合您情况的模型。

## 基本用法
模型可以通过两种方式加以利用：
- 使用代理时，可以动态指定模型。
- 独立运行- 可以直接调用模型（在代理循环之外）来执行文本生成、分类或提取等任务，而无需代理框架。

同一个模型接口适用于两种情况，这使您可以灵活地从简单的开始，并根据需要扩展到更复杂的基于代理的工作流。

### 初始化模型
在 LangChain 中使用独立模型的最简单方法是，使用您选择的聊天模型提供商init_chat_model初始化一个模型（示例如下）：

`pip install -U "langchain[openai]"`

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = "sk-..."

model = init_chat_model("gpt-4.1")

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = "sk-..."

model = init_chat_model("gpt-4.1")

In [ ]:
import os
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = "sk-..."

model = ChatOpenAI(model="gpt-4.1")

In [ ]:
response = model.invoke("Why do parrots talk?")

## 参数
- model
您要与服务提供商一起使用的特定型号的名称或标识符。

- api_key
用于向模型提供商进行身份验证的密钥。通常在您注册访问模型时颁发。通常通过设置来访问。

- temperature
控制模型输出的随机性。数值越高，响应越具创造性；数值越低，响应越具确定性。

- timeout
等待模型响应的最长时间（以秒为单位），超过此时间将取消请求。

- max_tokens
限制总数代币在响应中，有效地控制输出的长度。

- max_retries
如果由于网络超时或速率限制等问题导致请求失败，系统将尝试重新发送请求的最大次数。


使用 `init_chat_model`，以内联 `**kwargs` 的形式传递这些参数：


In [ ]:
model = init_chat_model(
    "claude-sonnet-4-5-20250929",
    # Kwargs passed to the model:
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
)

## 调用
必须调用聊天模型才能生成输出。有三种主要的调用方法，每种方法都适用于不同的使用场景。

### 调用
调用模型的最直接方法是使用invoke()单个消息或消息列表。



In [ ]:
response = model.invoke("Why do parrots have colorful feathers?")
print(response)

可以向模型提供消息列表来表示对话历史记录。每条消息都有一个角色，模型使用该角色来指示对话中消息的发送者。有关角色、类型和内容的更多详细信息，请参阅消息指南。


##### 字典格式

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

#### 消息对象

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications.")
]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

### 流
大多数模型都能在生成输出内容的同时进行流式传输。通过逐步显示输出，流式传输显著提升了用户体验，尤其是在处理较长的响应时。


调用`stream()`返回一个迭代器它会在生成过程中实时输出数据块。您可以使用循环来实时处理每个数据块：

In [ ]:
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

`invoke()` 会在模型生成完整响应后返回单个 `AIMessage`，而 `stream()` 则会返回多个 `AIMessageChunk` 对象，每个对象都包含输出文本的一部分。重要的是，流中的每个块都可以通过求和汇集成完整的信息：

In [ ]:
full = None  # None | AIMessageChunk
for chunk in model.stream("What color is the sky?"):
    full = chunk if full is None else full + chunk
    print(full.text)

# The
# The sky
# The sky is
# The sky is typically
# The sky is typically blue
# ...

print(full.content_blocks)
# [{"type": "text", "text": "The sky is typically blue..."}]

生成的消息可以像使用 `getMessage()` 生成的消息一样处理invoke()——例如，它可以聚合到消息历史记录中，并作为对话上下文传递回模型。

### 批
将一系列独立的模型请求批量处理，可以显著提高性能并降低成本，因为可以并行处理这些请求：


In [ ]:
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
for response in responses:
    print(response)

默认情况下，batch()只会返回整个批次的最终输出。如果您希望在每个输入生成完成后立即接收其输出，可以使用以下方式流式传输结果batch_as_completed()：

在使用 `batch()` 或 `batch_as_completed()` 处理大量输入时，可能需要控制并行调用的最大次数。这可以通过设置 `RunnableConfig `字典中的 `max_concurrency` 属性来实现。



### 工具调用
 模型可以请求调用工具来执行诸如从数据库获取数据、搜索网络或运行代码等任务。工具是以下各项的组合：
- 模式，包括工具名称、描述和/或参数定义（通常是 JSON 模式）
- 函数或协程执行。


要使你定义的工具可供模型使用，你必须使用绑定方法将它们绑定起来bind_tools()。在后续调用中，模型可以根据需要选择调用任何已绑定的工具。

某些模型提供商提供内置工具，可通过模型或调用参数启用（例如ChatOpenAI，ChatAnthropic）。有关详细信息，请参阅相应的提供商参考文档。

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."


model_with_tools = model.bind_tools([get_weather])  

response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

绑定用户自定义工具时，模型的响应包含执行工具的请求。如果模型与代理分开使用，则需要您执行请求的操作并将结果返回给模型以供后续推理使用。请注意，如果使用代理，代理循环将为您处理工具执行循环。
下面，我们将展示一些使用工具调用的常见方法。

##### Pydantic 模型提供最丰富的功能集，包括字段验证、描述和嵌套结构。




In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # Movie(title="Inception", year=2010, director="Christopher Nolan", rating=8.8)

##### TypedDict使用 Python 的内置类型系统提供了一种更简单的替代方案，非常适合不需要运行时验证的情况。



In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}

##### 为了获得最大的控制权或互操作性，您可以提供原始的 JSON Schema。

In [ ]:
import json

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

model_with_structure = model.with_structured_output(
    json_schema,
    method="json_schema",
)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, ...}

结构化输出的关键考虑因素：
- 方法参数：某些提供商支持不同的方法（'json_schema'，function_calling'，'json_mode'）
    - 'json_schema'通常指提供商提供的专用结构化输出功能
    - 'function_calling'通过强制按照给定模式调用工具来生成结构化输出
    - 'json_mode'这是某些提供商提供的先导功能——'json_schema'它可以生成有效的 JSON，但必须在提示符中描述模式。
- 包含原始数据：用于include_raw=True同时获取解析后的输出和原始 AI 消息。
- 验证：Pydantic 模型提供自动验证，而TypedDictJSON Schema 则需要手动验证。

#### 消息输出及解析后的结构

`AIMessage`为了访问响应元数据（例如标记计数） ，返回原始对象以及解析后的表示形式可能很有用。为此，请`include_raw=True`在调用时进行设置`with_structured_output`：

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  
response = model_with_structure.invoke("Provide details about the movie Inception")
response
# {
#     "raw": AIMessage(...),
#     "parsed": Movie(title=..., year=..., ...),
#     "parsing_error": None,
# }

#### 嵌套结构


In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

## 高级主题
​

### 多模态
某些模型可以处理并返回非文本数据，例如图像、音频和视频。您可以通过提供内容块将非文本数据传递给模型。

部分型号可以返回多模态数据作为响应的一部分。如果调用此方法，则结果AIMessage将包含具有多模态类型的内容块

In [ ]:
response = model.invoke("Create a picture of a cat")
print(response.content_blocks)
# [
#     {"type": "text", "text": "Here's a picture of a cat"},
#     {"type": "image", "base64": "...", "mime_type": "image/jpeg"},
# ]

### 推理
新型模型能够进行多步骤推理以得出结论。这涉及到将复杂问题分解成更小、更易于处理的步骤。

如果底层模型支持，你可以将这个推理过程展现出来，以便更好地理解模型是如何得出最终答案的。

In [ ]:
for chunk in model.stream("Why do parrots have colorful feathers?"):
    reasoning_steps = [r for r in chunk.content_blocks if r["type"] == "reasoning"]
    print(reasoning_steps if reasoning_steps else chunk.text)

In [ ]:
response = model.invoke("Why do parrots have colorful feathers?")
reasoning_steps = [b for b in response.content_blocks if b["type"] == "reasoning"]
print(" ".join(step["reasoning"] for step in reasoning_steps))

根据模型的不同，有时可以指定模型进行推理的投入程度。同样，也可以要求模型完全关闭推理功能。这可以通过设置推理的“层级”（例如，'low'或'high'）或整数令牌预算来实现。

详情请参阅集成页面或您相应聊天模型的参考资料。

### 本地模型
LangChain 支持在您自己的硬件上本地运行模型。这对于数据隐私至关重要、需要调用自定义模型或希望避免使用云端模型所产生的费用等场景非常有用。
Ollama是在本地运行模型最简便的方法之一。请在集成页面查看完整的本地集成列表。
​
### 提示缓存
许多服务提供商提供即时缓存功能，以减少重复处理相同令牌时的延迟和成本。这些功能可以是隐式的，也可以是显式的：
隐式提示缓存：如果请求命中缓存，服务提供商会自动将节省的成本传递给目标提供商。例如：OpenAI和Gemini（Gemini 2.5 及更高版本）。
显式缓存：提供商允许您手动指定缓存点，以便更好地控制或确保节省成本。例如：（ChatOpenAI通过prompt_cache_key），Anthropic 的AnthropicPromptCachingMiddleware选项cache_control，AWS Bedrock，Gemini。


### 服务器端工具的使用
一些提供商支持服务器端工具调用循环：模型可以与网络搜索、代码解释器和其他工具进行交互，并在一次对话中分析结果。
如果模型调用服务器端工具，则响应消息的内容将包含表示工具调用和结果的内容。访问响应的内容块将以与提供商无关的格式返回服务器端工具调用和结果：

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1-mini")

tool = {"type": "web_search"}
model_with_tools = model.bind_tools([tool])

response = model_with_tools.invoke("What was a positive news story from today?")
response.content_blocks

```json
[
    {
        "type": "server_tool_call",
        "name": "web_search",
        "args": {
            "query": "positive news stories today",
            "type": "search"
        },
        "id": "ws_abc123"
    },
    {
        "type": "server_tool_result",
        "tool_call_id": "ws_abc123",
        "status": "success"
    },
    {
        "type": "text",
        "text": "Here are some positive news stories from today...",
        "annotations": [
            {
                "end_index": 410,
                "start_index": 337,
                "title": "article title",
                "type": "citation",
                "url": "..."
            }
        ]
    }
]
```

这代表一次对话；没有像客户端工具调用那样需要传入的关联ToolMessage对象。

请查看您所用服务提供商的集成页面，了解可用的工具和使用详情。

##### 初始化并使用速率限制器

LangChain 内置了一个（可选的）限制器InMemoryRateLimiter。该限制器是线程安全的，可以在同一进程中由多个线程共享。



In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # 1 request every 10s
    check_every_n_seconds=0.1,  # Check every 100ms whether allowed to make a request
    max_bucket_size=10,  # Controls the maximum burst size.
)

model = init_chat_model(
    model="gpt-5",
    model_provider="openai",
    rate_limiter=rate_limiter  
)

### 基本 URL 或代理
对于许多聊天模型集成，您可以配置 API 请求的基本 URL，这样您就可以使用具有 OpenAI 兼容 API 的模型提供商或使用代理服务器。

许多模型提供商都提供与 OpenAI 兼容的 API（例如Together AI、vLLM）。您可以init_chat_model通过指定相应的base_url参数来使用这些提供商：

In [ ]:
model = init_chat_model(
    model="MODEL_NAME",
    model_provider="openai",
    base_url="BASE_URL",
    api_key="YOUR_API_KEY",
)

#### 代理配置

对于需要 HTTP 代理的部署，某些模型集成支持代理配置：


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4o",
    openai_proxy="http://proxy.example.com:8080"
)

### Log probabilities

logprobs某些模型可以通过在初始化模型时设置参数来配置为返回表示给定令牌可能性的令牌级日志概率：

In [ ]:
model = init_chat_model(
    model="gpt-4o",
    model_provider="openai"
).bind(logprobs=True)

response = model.invoke("Why do parrots talk?")
print(response.response_metadata["logprobs"])

### Token使用
许多模型提供程序会在调用响应中返回令牌使用信息。如果可用，此信息将包含在AIMessage相应模型生成的对象中。更多详细信息，请参阅消息指南。

您可以使用回调或上下文管理器来跟踪应用程序中各个模型的聚合令牌计数，如下所示：

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import UsageMetadataCallbackHandler

model_1 = init_chat_model(model="gpt-4o-mini")
model_2 = init_chat_model(model="claude-haiku-4-5-20251001")

callback = UsageMetadataCallbackHandler()
result_1 = model_1.invoke("Hello", config={"callbacks": [callback]})
result_2 = model_2.invoke("Hello", config={"callbacks": [callback]})
callback.usage_metadata

#### 上下文管理器

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import get_usage_metadata_callback

model_1 = init_chat_model(model="gpt-4o-mini")
model_2 = init_chat_model(model="claude-haiku-4-5-20251001")

with get_usage_metadata_callback() as cb:
    model_1.invoke("Hello")
    model_2.invoke("Hello")
    print(cb.usage_metadata)

```python

{
    'gpt-4o-mini-2024-07-18': {
        'input_tokens': 8,
        'output_tokens': 10,
        'total_tokens': 18,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    },
    'claude-haiku-4-5-20251001': {
        'input_tokens': 8,
        'output_tokens': 21,
        'total_tokens': 29,
        'input_token_details': {'cache_read': 0, 'cache_creation': 0}
    }
}
```

### 调用配置
调用模型时，可以通过config参数使用RunnableConfig字典传递额外的配置信息。这可以实现对执行行为、回调和元数据跟踪的运行时控制。

In [ ]:
response = model.invoke(
    "Tell me a joke",
    config={
        "run_name": "joke_generation",      # Custom name for this run
        "tags": ["humor", "demo"],          # Tags for categorization
        "metadata": {"user_id": "123"},     # Custom metadata
        "callbacks": [my_callback_handler], # Callback handlers
    }
)

### 可配置模型
您还可以通过指定来创建运行时可配置模型configurable_fields。如果您不指定模型值，则默认情况下'model'和'model_provider'将可配置。

In [ ]:
from langchain.chat_models import init_chat_model

configurable_model = init_chat_model(temperature=0)

configurable_model.invoke(
    "what's your name",
    config={"configurable": {"model": "gpt-5-nano"}},  # Run with GPT-5-Nano
)
configurable_model.invoke(
    "what's your name",
    config={"configurable": {"model": "claude-sonnet-4-5-20250929"}},  # Run with Claude
)

#### 可配置模型，具有默认值


我们可以创建一个可配置模型，设置默认模型值，指定哪些参数可配置，并为可配置参数添加前缀：




In [ ]:
first_model = init_chat_model(
        model="gpt-4.1-mini",
        temperature=0,
        configurable_fields=("model", "model_provider", "temperature", "max_tokens"),
        config_prefix="first",  # Useful when you have a chain with multiple models
)

first_model.invoke("what's your name")


first_model.invoke(
    "what's your name",
    config={
        "configurable": {
            "first_model": "claude-sonnet-4-5-20250929",
            "first_temperature": 0.5,
            "first_max_tokens": 100,
        }
    },
)

#### 使用可配置模型进行声明式编程
我们可以在可配置模型上调用声明式操作，例如bind_tools,with_structured_output，with_configurable等等，并且以与常规实例化聊天模型对象相同的方式链接可配置模型。

In [ ]:
from pydantic import BaseModel, Field


class GetWeather(BaseModel):
    """Get the current weather in a given location"""

    location: str = Field(..., description="The city and state, e.g. San Francisco, CA")


class GetPopulation(BaseModel):
    """Get the current population in a given location"""

    location: str = Field(..., description="The city and state, e.g. San Francisco, CA")


model = init_chat_model(temperature=0)
model_with_tools = model.bind_tools([GetWeather, GetPopulation])

model_with_tools.invoke(
    "what's bigger in 2024 LA or NYC", config={"configurable": {"model": "gpt-4.1-mini"}}
).tool_calls


model_with_tools.invoke(
    "what's bigger in 2024 LA or NYC",
    config={"configurable": {"model": "claude-sonnet-4-5-20250929"}},
).tool_calls